In [ ]:
import json 
from src.llm import LLM
from src.prompt import *
from src.utils import iterate_schelling_data, coordination_index

prompt_schema = Level0  # prompting technique

In [40]:
# Get the data
with open('./data/schelling.jsonl', 'r') as file:
    data = json.load(file)

# Get the combinations of problems
problems = iterate_schelling_data(data)

In [28]:
# model = lambda x: random.choice(['<answer>A</answer>', '<answer>B</answer>'])  # Mock model for demonstration
trials = 3  # Number of trials for each problem
model = LLM(model_id="meta-llama/Llama-3.2-1B",
            num_return_sequences=3)

# Run the experiments
responses = {}
for idx in problems:
    responses[idx] = {}
    for problem in problems[idx]:
        if problem not in responses[idx]:
            responses[idx][problem] = []
            
        prompt = Level0.prefix + problem + Level0.suffix
        for _ in range(trials):
            # Here we call the model and get the reponse
            # 
            response = model(prompt)  # fake reponse to test the code
            responses[idx][problem].append(response)
        

In [37]:
logs = []
for idx in responses:
    for problem in responses[idx]:
        log = {
            'idx': idx,
            'prompt': problem,
            "responses": responses[idx][problem]
        }
    
        logs.append(log)
    
with open('./logs/schelling_responses.jsonl', 'w') as f:
    json.dump(logs, f, indent=2)

In [38]:
# Clean the data 
for idx in responses:
    for response in responses[idx]:
        try:
            responses[idx][response] = [r.replace('<answer>', '').replace('</answer>', '').strip() for r in responses[idx][response]]
        except:
            # Delete the response if it fails to clean
            del responses[idx][response]

In [23]:
# Compute the coordination index
results = []
for idx in responses:
    for response in responses[idx]:
        if responses[idx][response]:
            c_index, nc_index = coordination_index(responses[idx][response])
            
            result = {
                "idx": idx,
                "problem": response,
                "coordination_index": c_index,
                "normalised_coordination_index": nc_index
            }
            
            results.append(result)
              
# Write the results to a file
with open("./results/schelling_results.jsonl", "a") as f:
    json.dump(results, f, indent=2)
    